# Paper-Style Results Notebook: Intralingual Cultural Adaptation

This notebook is structured like an experimental results section for an NLP paper.

**Task:** Adapt text from source culture to target culture (Indian subregions), preserving intent while localizing context.

## 1. Setup and Reproducibility
Set paths below. The expected pipeline outputs are:
- `adaptations.jsonl`
- `metrics.csv`
- `summary.json`

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

RUN_DIR = Path('../outputs/run1')  # change if needed
ADAPT_PATH = RUN_DIR / 'adaptations.jsonl'
METRICS_PATH = RUN_DIR / 'metrics.csv'
SUMMARY_PATH = RUN_DIR / 'summary.json'

print('Run dir:', RUN_DIR.resolve())
print('Exists:', RUN_DIR.exists())

## 2. Load Outputs

In [ ]:
if METRICS_PATH.exists():
    metrics_df = pd.read_csv(METRICS_PATH)
else:
    metrics_df = pd.DataFrame()

if SUMMARY_PATH.exists():
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
else:
    summary = {}

adapt_rows = []
if ADAPT_PATH.exists():
    for line in ADAPT_PATH.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line:
            adapt_rows.append(json.loads(line))
adapt_df = pd.DataFrame(adapt_rows)

print('metrics:', metrics_df.shape)
print('adaptations:', adapt_df.shape)
print('summary keys:', list(summary.keys()))

## 3. Main Aggregate Results

In [ ]:
if metrics_df.empty:
    print('No metrics found. Run the pipeline first.')
else:
    metric_cols = [
        'content_similarity','target_culture_signal','adaptation_depth',
        'lexical_shift','stereotype_risk','composite_score'
    ]
    available = [c for c in metric_cols if c in metrics_df.columns]
    table = metrics_df[available].agg(['mean','std','min','max']).T
    display(table.round(4))

## 4. Culture-Pair Transfer Matrix
Average composite score by source-target culture pair.

In [ ]:
if metrics_df.empty or 'composite_score' not in metrics_df.columns:
    print('Composite score unavailable.')
else:
    pair_matrix = metrics_df.pivot_table(
        index='source_culture',
        columns='target_culture',
        values='composite_score',
        aggfunc='mean'
    )
    display(pair_matrix.round(3))

    if HAS_MPL:
        ax = pair_matrix.plot(kind='bar', figsize=(11,4))
        ax.set_title('Composite Score by Culture Pair')
        ax.set_ylabel('Mean Composite Score')
        plt.tight_layout()
        plt.show()

## 5. Genre-Wise Performance

In [ ]:
if metrics_df.empty or 'genre' not in metrics_df.columns:
    print('Genre column unavailable.')
else:
    cols = [c for c in ['content_similarity','target_culture_signal','adaptation_depth','composite_score'] if c in metrics_df.columns]
    genre_table = metrics_df.groupby('genre')[cols].mean().sort_values(by=cols[-1], ascending=False)
    display(genre_table.round(4))

## 6. Human Evaluation Merge (Optional)
If you annotated `eval/human_eval_sheet_template.csv`, merge and compare with automatic metrics.

In [ ]:
HUMAN_PATH = Path('../eval/human_eval_sheet_template.csv')  # replace with filled annotations

if metrics_df.empty or not HUMAN_PATH.exists():
    print('Need metrics and human eval CSV for merge analysis.')
else:
    human_df = pd.read_csv(HUMAN_PATH)
    left_id = 'item_id' if 'item_id' in metrics_df.columns else 'id'
    right_id = 'item_id' if 'item_id' in human_df.columns else 'id'
    if left_id not in metrics_df.columns or right_id not in human_df.columns:
        raise ValueError('Could not find ID columns for merge. Expected id/item_id.')
    merged = metrics_df.merge(human_df, left_on=left_id, right_on=right_id, how='inner')
    print('merged rows:', merged.shape[0])

    score_cols = [c for c in ['authenticity_1to5','faithfulness_1to5','coherence_1to5','safety_1to5','adaptation_depth_1to5'] if c in merged.columns]
    for c in score_cols:
        merged[c] = pd.to_numeric(merged[c], errors='coerce')

    auto_cols = [c for c in ['content_similarity','target_culture_signal','adaptation_depth','composite_score'] if c in merged.columns]
    if score_cols and auto_cols:
        corr = merged[score_cols + auto_cols].corr(numeric_only=True)
        display(corr.loc[score_cols, auto_cols].round(3))
    else:
        print('Not enough columns for correlation analysis.')

## 7. Qualitative Examples (Best / Worst)

In [ ]:
if metrics_df.empty or adapt_df.empty or 'composite_score' not in metrics_df.columns:
    print('Need metrics + adaptations for qualitative view.')
else:
    id_col = 'id'
    q = metrics_df[[id_col, 'composite_score']].merge(adapt_df, on=id_col, how='left')

    print('--- TOP 3 ---')
    for _, r in q.sort_values('composite_score', ascending=False).head(3).iterrows():
        print(f"\n[{r[id_col]}] score={r['composite_score']:.3f} ({r['source_culture']} -> {r['target_culture']})")
        print('SOURCE :', r.get('source_text', '')[:220])
        print('ADAPTED:', r.get('adapted_text', '')[:220])

    print('\n--- BOTTOM 3 ---')
    for _, r in q.sort_values('composite_score', ascending=True).head(3).iterrows():
        print(f"\n[{r[id_col]}] score={r['composite_score']:.3f} ({r['source_culture']} -> {r['target_culture']})")
        print('SOURCE :', r.get('source_text', '')[:220])
        print('ADAPTED:', r.get('adapted_text', '')[:220])

## 8. Report-Ready Summary Paragraph
This cell creates a concise paragraph you can paste into your paper/report.

In [ ]:
if metrics_df.empty:
    print('No results yet.')
else:
    n = len(metrics_df)
    c = metrics_df['composite_score'].mean() if 'composite_score' in metrics_df.columns else np.nan
    p = metrics_df['content_similarity'].mean() if 'content_similarity' in metrics_df.columns else np.nan
    g = metrics_df['target_culture_signal'].mean() if 'target_culture_signal' in metrics_df.columns else np.nan
    d = metrics_df['adaptation_depth'].mean() if 'adaptation_depth' in metrics_df.columns else np.nan

    paragraph = (
        f"Across {n} samples, the system achieved an average composite score of {c:.3f}. "
        f"Content preservation remained strong (mean={p:.3f}), while target-culture grounding was "
        f"moderate-to-high (mean={g:.3f}). Adaptation depth (mean={d:.3f}) indicates scenario-level "
        f"localization beyond simple lexical substitutions on a substantial portion of samples."
    )
    print(paragraph)